## Imports

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

## Set current dir to project root dir

In [2]:
def find_project_root():
    """Walk up from CWD until we find the project root."""
    markers = [".git", "Makefile", "renv.lock", ".Rprofile"]
    path = Path.cwd()
    while path != path.parent:
        if any((path / m).exists() for m in markers):
            return path
        path = path.parent
    raise FileNotFoundError("Could not find project root")

os.chdir(find_project_root())

## Configuration

In [3]:
stimuli_dir    = "./ds006018_per_stimuli"
target_csv     = "task-flanker_Stimulus_S2.csv"
target_channel = "FC1"

output_csv     = "./EDA/Flanker_stimulus_FC1_channel.csv"

## Step 1: Find all subjects that have this file

In [4]:
all_csv_paths = sorted(Path(stimuli_dir).rglob(target_csv))

print("=== DISCOVERY ===")
print(f"Target: {target_csv}")
print(f"Channel: {target_channel}")
print(f"Found {len(all_csv_paths)} subjects\n")

if len(all_csv_paths) == 0:
    raise FileNotFoundError("No files found! Check stimuli_dir and target_csv.")

# Extract original subject IDs from paths
original_subject_ids = [p.parent.name for p in all_csv_paths]

=== DISCOVERY ===
Target: task-flanker_Stimulus_S2.csv
Channel: FC1
Found 62 subjects



## Step 2: Loop through each subject, extract channel, average if needed

In [5]:
subject_curves = {}
time_vec = None
n_epochs_per_subject = []

for i, path_i in enumerate(all_csv_paths):
    orig_id = original_subject_ids[i]
    df_i = pd.read_csv(path_i)

    # --- Sanity checks ---
    if target_channel not in df_i.columns:
        print(f"WARNING: Channel {target_channel} not found in {orig_id} — skipping.")
        continue

    if not {"time", "epoch"}.issubset(df_i.columns):
        print(f"WARNING: Missing 'time' or 'epoch' column in {orig_id} — skipping.")
        continue

    # --- How many epochs does this subject have? ---
    n_epochs = df_i["epoch"].nunique()
    n_epochs_per_subject.append(n_epochs)

    if n_epochs == 1:
        # Single epoch — take the channel column directly
        df_i = df_i.sort_values("time")
        mean_curve = df_i[target_channel].to_numpy()

        if time_vec is None:
            time_vec = df_i["time"].to_numpy()

    else:
        # Multiple epochs — reshape and average across trials
        trial_matrix = df_i.pivot(index="time", columns="epoch",
                                  values=target_channel)
        mean_curve = trial_matrix.mean(axis=1).to_numpy()

        if time_vec is None:
            time_vec = trial_matrix.index.to_numpy(dtype=float)

    subject_curves[orig_id] = mean_curve

    averaged = " (averaged)" if n_epochs > 1 else ""
    print(f"  [{i+1}/{len(all_csv_paths)}] {orig_id} — {n_epochs} epoch(s){averaged}")

print(f"\nProcessed: {len(subject_curves)} subjects")
print(f"  Single-epoch subjects: {sum(n == 1 for n in n_epochs_per_subject)}")
print(f"  Multi-epoch subjects:  {sum(n > 1 for n in n_epochs_per_subject)} (averaged)")

  [1/62] sub-001 — 1 epoch(s)
  [2/62] sub-002 — 1 epoch(s)
  [3/62] sub-003 — 1 epoch(s)
  [4/62] sub-004 — 1 epoch(s)
  [5/62] sub-005 — 1 epoch(s)
  [6/62] sub-006 — 1 epoch(s)
  [7/62] sub-007 — 2 epoch(s) (averaged)
  [8/62] sub-008 — 1 epoch(s)
  [9/62] sub-009 — 1 epoch(s)
  [10/62] sub-010 — 1 epoch(s)
  [11/62] sub-011 — 1 epoch(s)
  [12/62] sub-012 — 1 epoch(s)
  [13/62] sub-013 — 1 epoch(s)
  [14/62] sub-014 — 1 epoch(s)
  [15/62] sub-016 — 1 epoch(s)
  [16/62] sub-017 — 1 epoch(s)
  [17/62] sub-019 — 1 epoch(s)
  [18/62] sub-021 — 1 epoch(s)
  [19/62] sub-022 — 2 epoch(s) (averaged)
  [20/62] sub-024 — 1 epoch(s)
  [21/62] sub-025 — 1 epoch(s)
  [22/62] sub-027 — 1 epoch(s)
  [23/62] sub-028 — 1 epoch(s)
  [24/62] sub-029 — 2 epoch(s) (averaged)
  [25/62] sub-030 — 1 epoch(s)
  [26/62] sub-031 — 1 epoch(s)
  [27/62] sub-032 — 1 epoch(s)
  [28/62] sub-033 — 1 epoch(s)
  [29/62] sub-034 — 1 epoch(s)
  [30/62] sub-035 — 1 epoch(s)
  [31/62] sub-036 — 1 epoch(s)
  [32/62] sub-0

## Step 3: Combine subjects (keep original IDs)

In [6]:
Y_mat = np.column_stack(list(subject_curves.values()))
n_subjects = Y_mat.shape[1]

print("=== SUBJECTS INCLUDED ===")
print(", ".join(subject_curves.keys()))
print(f"Total: {n_subjects}")
print("=========================")

metadata_df = pd.DataFrame({
    "subject_id": list(subject_curves.keys()),
    "n_epochs":   n_epochs_per_subject[:n_subjects]
})

print("\n=== EPOCH COUNTS ===")
print(metadata_df.to_string(index=False))
print("====================")

=== SUBJECTS INCLUDED ===
sub-001, sub-002, sub-003, sub-004, sub-005, sub-006, sub-007, sub-008, sub-009, sub-010, sub-011, sub-012, sub-013, sub-014, sub-016, sub-017, sub-019, sub-021, sub-022, sub-024, sub-025, sub-027, sub-028, sub-029, sub-030, sub-031, sub-032, sub-033, sub-034, sub-035, sub-036, sub-037, sub-038, sub-039, sub-040, sub-041, sub-042, sub-043, sub-046, sub-047, sub-048, sub-049, sub-050, sub-052, sub-053, sub-055, sub-057, sub-058, sub-059, sub-060, sub-061, sub-062, sub-063, sub-064, sub-065, sub-066, sub-068, sub-069, sub-070, sub-071, sub-072, sub-073
Total: 62

=== EPOCH COUNTS ===
subject_id  n_epochs
   sub-001         1
   sub-002         1
   sub-003         1
   sub-004         1
   sub-005         1
   sub-006         1
   sub-007         2
   sub-008         1
   sub-009         1
   sub-010         1
   sub-011         1
   sub-012         1
   sub-013         1
   sub-014         1
   sub-016         1
   sub-017         1
   sub-019         1
   sub-

## Step 4: Export

In [7]:
final_df = pd.DataFrame(Y_mat, columns=list(subject_curves.keys()))
final_df.insert(0, "time", time_vec)

print(f"Final matrix: {final_df.shape[0]} time points × {n_subjects} subjects\n")

final_df.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

metadata_df.to_csv("./EDA/subject_metadata.csv", index=False)
print("Saved: ./EDA/subject_metadata.csv")

print(f"\n*** Done! All {n_subjects} subjects with S2 flanker data. ***")

Final matrix: 501 time points × 62 subjects

Saved: ./EDA/Flanker_stimulus_FC1_channel.csv
Saved: ./EDA/subject_metadata.csv

*** Done! All 62 subjects with S2 flanker data. ***
